# **OCR Pruebas**

## **Updating for new docs**

## **Función actualizada**

In [1]:
import os
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

In [3]:
import pytesseract

# 1. Ruta al ejecutable (ajusta según tu carpeta)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'

# 2. Asegurar que Python vea los datos de idioma
os.environ['TESSDATA_PREFIX'] = r'C:\Program Files\Tesseract-OCR\tessdata'

In [4]:
def load_extract_split_embeddings(
    directory_path,
    persist_directory="./chroma_db",
    chunk_size=1200, # Aumentado para dar más contexto legal por bloque
    chunk_overlap=200,
    model_name="qwen2.5:7b",   #"llama3.2",
    create_new=True
):
    # 1. Cargar y Procesar con OCR
    # Usamos Unstructured para manejar PDFs escaneados
    all_chunks = []
    
    for filename in os.listdir(directory_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(directory_path, filename)
            print(f"Procesando con OCR: {filename}...")
            
            # El modo 'elements' extrae tablas y títulos con jerarquía
            loader = UnstructuredPDFLoader(
                file_path, 
                strategy="ocr_only", # Obligamos a OCR por ser escaneados
                languages=["spa"]    # Muy importante para las tildes y 'ñ'
            )
            docs = loader.load()
            
            # 2. División conservando Metadatos Críticos
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
            
            # Añadimos metadatos de origen para la "Sorpresa" de descarga
            for doc in docs:
                doc.metadata["source_file"] = filename
                doc.metadata["full_path"] = file_path
                
            file_chunks = splitter.split_documents(docs)
            all_chunks.extend(file_chunks)

    # 3. Embeddings e Indexación
    embeddings = OllamaEmbeddings(model=model_name)

    if create_new and os.path.exists(persist_directory):
        import shutil
        shutil.rmtree(persist_directory) # Limpiamos para evitar duplicados

    vectorstore = Chroma.from_documents(
        documents=all_chunks,
        embedding=embeddings,
        persist_directory=persist_directory
    )
    
    print(f"Éxito: {len(all_chunks)} fragmentos indexados.")
    return vectorstore

In [5]:
# --- Configuración del Sistema de Agentes ---

llm = ChatOllama(model="qwen2.5:7b", temperature=0)
vector_store = load_extract_split_embeddings("./Contratos", create_new=True)
retriever = vector_store.as_retriever(search_kwargs={"k": 8})

Procesando con OCR: C01_Contrato_Prueba-01.pdf...
Procesando con OCR: C02_Contrato_Colima-02.pdf...
Éxito: 46 fragmentos indexados.


In [6]:
# Obtenemos todos los metadatos de la base de datos
datos = vector_store.get()
metadatos = datos['metadatas']

# Contamos cuántos fragmentos hay de cada archivo
conteo = {}
for meta in metadatos:
    fuente = meta.get('source_file', 'Sin nombre')
    conteo[fuente] = conteo.get(fuente, 0) + 1

print("📊 RESUMEN DE INDEXACIÓN:")
for archivo, cantidad in conteo.items():
    print(f"- {archivo}: {cantidad} fragmentos indexados")

📊 RESUMEN DE INDEXACIÓN:
- C01_Contrato_Prueba-01.pdf: 28 fragmentos indexados
- C02_Contrato_Colima-02.pdf: 18 fragmentos indexados


In [7]:
# Ajuste en el Prompt para la "Sorpresa"
template_base = """
SISTEMA DE AUDITORÍA LEGAL EXPERTO: ACCESO AUTORIZADO.
Estás visualizando fragmentos extraídos mediante OCR de los contratos de Vertiche. 
Tu única tarea es extraer información de la TRANSCRIPCIÓN del contexto proporcionado.

GUÍA DE CONCEPTOS:
- ARRENDADOR: Es casi siempre la PRIMERA entidad o persona mencionada en el primer párrafo del contrato. El DUEÑO de la propiedad (quien recibe la renta).
- ARRENDATARIO: Es "COMERCIAL IAC, S. A. DE C. V." la SEGUNDA entidad mencionada en el primer párrafo del contrato.

OBJETIVO: Respuesta ESTRUCTURADA. El dato principal debe ser visible al instante.

INSTRUCCIONES DE FORMATO (ESTRICTO):
1. **Dato Específico**: La primera línea debe ser UNICAMENTE el nombre, monto o fecha solicitado.
2. **Detalles Relevantes**: Lista de puntos con nombres de beneficiarios, representantes o condiciones clave.
3. **Contexto Legal**: Párrafo breve con obligaciones relacionadas.
4. **Fuente**: Nombre del archivo al final.

Sigue EXACTAMENTE este formato:

USER: ¿query?
AI: 
Dato especifico: [Dato concreto]
- Justificación
- Detalles
Información legal adicional relevante.


REGLAS DE EXTRACCIÓN:
1. PROHIBIDO resumir nombres propios, fechas o montos. Transcríbelos tal cual.
2. Si mencionan beneficiarios o terceros, lístalos TODOS.
3. Si el usuario pide el Contrato X, ignora fragmentos que no sean de la fuente X.
4. Prioriza la exhaustividad de los datos (nombres/cifras) sobre la brevedad, pero mantén una estructura limpia y facil de leer.
5. NUNCA digas "No tengo acceso". Si el contexto está presente, tú tienes el acceso.

### INSTRUCCIÓN FINAL PARA EL MODELO:

CONTEXTO:
{context}

PREGUNTA:
{question}

RECUERDA: 
- Si la pregunta pide una "Cláusula Novena", busca tanto "Novena" como "9" como "IX".
- No respondas nada que no esté en el contexto.
- Si el monto en número no coincide con el monto en letra, escribe SOLAMENTE el de LETRA.

PENSAMIENTO PASO A PASO:
1. Identificar contrato solicitado. 
2. Filtrar fragmentos por fuente. 
3. Extraer nombres y montos escritos con letra.

RESPUESTA DETALLADA:
AI:
"""

prompt = ChatPromptTemplate.from_template(template_base)

def format_docs(docs):
    # Aquí es donde ocurre la magia de las fuentes
    formatted = ""
    for d in docs:
        formatted += f"\n[Fuente: {d.metadata.get('source_file')}]\n{d.page_content}\n"
    return formatted



import os
from thefuzz import fuzz, process
from langchain_core.runnables import RunnableLambda

def buscador_dinamico(query_input):
    # 1. Limpiar la pregunta
    query_text = query_input.get("question", str(query_input)) if isinstance(query_input, dict) else str(query_input)
    q_lower = query_text.lower()
    
    # 2. Obtener lista de archivos REALES en la carpeta
    ruta_contratos = "./Contratos"
    archivos_reales = [f for f in os.listdir(ruta_contratos) if f.endswith(".pdf")]
    
    if not archivos_reales:
        return "Error: No hay PDFs en la carpeta /Contratos"

    # 3. EVALUACIÓN DE TODA LA LISTA (Para encontrar el máximo score)
    # Comparamos la pregunta contra cada nombre de archivo disponible
    # mapeo: { 'nombre_limpio': 'Nombre_Real_Archivo.pdf' }
    nombres_para_comparar = {f.lower().replace(".pdf", ""): f for f in archivos_reales}
    
    # Usamos token_set_ratio para que el orden no importe
    mejor_match, score = process.extractOne(
        q_lower, 
        list(nombres_para_comparar.keys()), 
        scorer=fuzz.token_set_ratio
    )
    # process.extractOne recorre TODA la lista y nos da el mejor de todos
#    mejor_match, score = process.extractOne(q_lower, list(nombres_para_comparar.keys()), scorer=fuzz.partial_ratio)

    filtro = None
    # 4. APLICACIÓN DEL GANADOR
    # Umbral de 70 para evitar falsos positivos en mil contratos
    if score >= 70:
        archivo_ganador = nombres_para_comparar[mejor_match]
        filtro = {"source_file": archivo_ganador}
        print(f"ARCHIVO: '{archivo_ganador}' con Score: {score}%")
    else:
        print(f"Ningún contrato superó el umbral (Mejor: {mejor_match} con {score}%). Buscando en todos.")

    # 5. Ejecutar búsqueda en ChromaDB
    if filtro:
        # k=10 para traer suficiente contexto del contrato específico
        docs = vector_store.similarity_search(query_text, k=15, filter=filtro)
    else:
        # k=15 si es búsqueda general para cubrir más terreno
        docs = vector_store.similarity_search(query_text, k=15)
        
    return format_docs(docs)

# RECUERDA RE-EJECUTAR LA CHAIN DESPUÉS DE CAMBIAR LA FUNCIÓN
chain = (
    {"context": RunnableLambda(buscador_dinamico), "question": RunnablePassthrough()}
    | prompt
    | llm
)






'''chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
)'''

'chain = (\n    {"context": retriever | format_docs, "question": RunnablePassthrough()}\n    | prompt\n    | llm\n)'

### Definir Herramientas

In [8]:
from langchain.tools import tool

# --- HERRAMIENTA 1: Búsqueda y Análisis (Tu Chain) ---
def tool_analizar_contrato(query: str):
    """Útil para buscar datos específicos, fechas o cláusulas dentro de los documentos."""
    return chain.invoke(query).content

# --- HERRAMIENTA 2: Filtrado y Listado ---
def tool_filtrar_datos(query: str):
    """Útil para listar contratos por ciudad o características (ej. 'Contratos en Monterrey')."""
    # Buscamos los documentos crudos para extraer metadatos
    docs = retriever.invoke(query)
    resultados = "Contratos encontrados:\n"
    for d in docs:
        resultados += f"- Archivo: {d.metadata.get('source_file')} | Ubicación: {d.metadata.get('full_path')}\n"
    return resultados

# --- HERRAMIENTA 3: Generación de Borradores ---
def tool_generar_borrador(detalles: str):
    """Útil para redactar nuevas cláusulas basadas en el estilo legal de Vertiche."""
    estilo = retriever.invoke("cláusulas estándar y formato")
    contexto_estilo = format_docs(estilo)
    
    prompt_gen = ChatPromptTemplate.from_template("""
    Eres un abogado de Vertiche. Usa este estilo: {estilo}
    Redacta lo siguiente: {detalles}
    """)
    
    # Llamada directa al LLM para creación
    chain_gen = prompt_gen | llm
    return chain_gen.invoke({"estilo": contexto_estilo, "detalles": detalles}).content

### Lista de Tools y configurar el Agente

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage

# --- 1. CONFIGURACIÓN DE TOOLS (Ya las tienes, solo asegúrate de que existan) ---
tools_map = {
    "Analizar": tool_analizar_contrato,
    "Filtrar": tool_filtrar_datos,
    "Generar": tool_generar_borrador
}

# --- 2. FUNCIÓN DEL AGENTE "SMART" ---
def smart_agent(pregunta):
    print(f"Pensando en la respuesta para: {pregunta}")
    
    # El prompt le dice al modelo qué herramientas tiene
    system_prompt = f"""Eres el asistente de SmartLease AI para Vertiche. 
    Tienes acceso a estas herramientas: {list(tools_map.keys())}.
    
    Para responder, primero di qué herramienta necesitas usar. 
    Si no necesitas herramientas, responde directamente."""
    
    # 1. El modelo decide qué hacer
    respuesta_inicial = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=pregunta)
    ]).content
    
    # 2. Lógica de ejecución de herramientas
    tool_found = False
    for name in tools_map.keys():
        if name.lower() in respuesta_inicial.lower():
            print(f"Usando herramienta: {name}...")
            resultado_tool = tools_map[name](pregunta)
            
            # 3. Respuesta final combinada
            respuesta_final = llm.invoke([
                SystemMessage(content="Genera una respuesta final basada en este resultado."),
                HumanMessage(content=f"Resultado de la herramienta: {resultado_tool}")
            ]).content
            return respuesta_final
            tool_found = True
            
    if not tool_found:
        return respuesta_inicial


### **Pruebas de funcionamiento**

In [11]:
# --- 3. PRUEBA ---
resultado = smart_agent("¿Quiénes son el arrendador y arrendatario del contrato de Colima?")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: ¿Quiénes son el arrendador y arrendatario del contrato de Colima?
Usando herramienta: Analizar...
ARCHIVO: 'C02_Contrato_Colima-02.pdf' con Score: 81%

RESPUESTA:
Basándome en el resultado proporcionado, la información clave del contrato de arrendamiento en Colima es la siguiente:

- Arrendador: C. EL ARRENDADOR
- Arrendatario principal: Apoderado Legal de Comercial IAC, S.A. de C.V.
- Representante del arrendatario: C. EL ARRENDATARIO 
- Obligado solidario: C. EL OBLIGADO SOLIDARIO

Esta estructura implica que el contrato involucra a varias partes y posiblemente se ha establecido una responsabilidad compartida o solidaria en caso de incumplimiento del acuerdo.


In [129]:
# --- 3. PRUEBA ---
resultado = smart_agent("Extrae la cláusula del periodo de gracia del contrato C01_Contrato")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Extrae la cláusula del periodo de gracia del contrato C01_Contrato
Usando herramienta: Analizar...
ARCHIVO: 'C01_Contrato_Prueba-01.pdf' con Score: 71%

RESPUESTA:
La cláusula del periodo de gracia del contrato C01_Contrato establece los siguientes términos:

* El periodo de gracia dura 120 días, desde el 1 de julio de 2025 hasta el 31 de octubre de 2025.
* Durante este período, el arrendatario no tiene obligación de pagar renta.
* El objetivo del periodo de gracia es que el arrendatario realice las adecuaciones necesarias para el buen funcionamiento del comercio en el inmueble.
* Una vez transcurrido el periodo de gracia, el arrendatario comenzará a pagar la renta mensual correspondiente, tal como se especifica en la cláusula tercera que antecede.

Es importante tener en cuenta que este periodo de gracia es una oportunidad para el arrendatario realizar las mejoras necesarias en el inmueble y evitar posibles problemas o sanciones por no cumplir con sus ob

In [ ]:
resultado = smart_agent("Extrae la renta del contrato de Colima?")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Extrae la renta del contrato de Colima?
Usando herramienta: Analizar...
ARCHIVO: 'C02_Contrato_Colima-02.pdf' con Score: 81%

RESPUESTA:
La respuesta final es:

Tres mil pesos 00/100 M.N.


In [148]:
resultado = smart_agent("Extrae la renta del contrato C01?")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Extrae la renta del contrato C01?
Usando herramienta: Analizar...
ARCHIVO: 'C01_Contrato_Prueba-01.pdf' con Score: 71%

RESPUESTA:
La renta del contrato C01 es de $29,703.76. Esto significa que el valor establecido para la prestación o servicio en cuestión es de aproximadamente $29,703.76. Si necesitas más información o contexto sobre este contrato, como la fecha de inicio o la duración del acuerdo, sería útil obtener más detalles para poder evaluar completamente la situación.


In [143]:
resultado = smart_agent("¿Cuáles son los intereses moratorios del contrato de Colima?")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: ¿Cuáles son los intereses moratorios del contrato de Colima?
Usando herramienta: Analizar...
ARCHIVO: 'C02_Contrato_Colima-02.pdf' con Score: 81%

RESPUESTA:
En resumen, según el contrato de Colima, se aplican intereses moratorios del 2% mensual más el impuesto al valor agregado (IVA) sobre la cantidad de renta generada mensualmente. Esto significa que los propietarios deben pagar un interés adicional del 2% sobre la renta mensual, más el IVA correspondiente, como forma de compensación por el retraso en el pago de la renta.


In [152]:
resultado = smart_agent("Extrae la cláusula de vigencia del contrato C01?")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Extrae la cláusula de vigencia del contrato C01?
Usando herramienta: Filtrar...

RESPUESTA:
Basado en el resultado proporcionado, parece que se han encontrado varios contratos guardados en un directorio llamado "Contratos". Sin embargo, hay algunas observaciones y recomendaciones que puedo hacer:

1. **Repetición de archivos**: Es notable ver que algunos archivos se repiten varias veces con la misma ubicación. Esto podría ser debido a una error en la búsqueda o a que los archivos se han copiado accidentalmente.
2. **Falta de información**: Aunque se han encontrado contratos, no hay mucha información adicional disponible sobre ellos, como fecha de creación, tipo de contrato, o contenido específico.
3. **Posibilidad de errores**: La repetición de archivos y la falta de información pueden indicar que hubo un error en la búsqueda o en la organización de los archivos.

Para abordar esto, te recomiendo lo siguiente:

1. Verificar la ubicación de los archivos y 

In [147]:
# Prueba de fuego
resultado = smart_agent("Contratos que cuentan con intereses moratorios")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Contratos que cuentan con intereses moratorios
Usando herramienta: Filtrar...

RESPUESTA:
**Análisis de los Contratos Encontrados**

Se han encontrado varios contratos guardados en el directorio "./Contratos". A continuación, se presentan los detalles de cada archivo:

*   **C01_Contrato_Prueba-01.pdf**: Este archivo se encuentra dos veces en el directorio. Es posible que haya sido copiado accidentalmente o que exista una versión original con un nombre diferente.
*   **C02_Contrato_Colima-02.pdf**: Este archivo también se encuentra varias veces en el directorio, lo que sugiere que puede haber sido copiado o que existe una versión original con un nombre diferente.

**Recomendaciones**

1.  Verificar la autenticidad de los archivos y eliminar las versiones duplicadas para evitar confusiones.
2.  Revisar los nombres de los archivos para asegurarse de que sean coherentes y no tengan errores de ortografía o gramática.
3.  Considerar la posibilidad de crear una

In [17]:
resultado = smart_agent("Genera una cláusula de publicidad usando el estilo de los contratos de tu base de datos")
print(f"\nRESPUESTA:\n{resultado}")

Pensando en la respuesta para: Genera una cláusula de publicidad usando el estilo de los contratos de tu base de datos
Usando herramienta: Generar...

RESPUESTA:
La respuesta final es:

**CLÁUSULA DE PUBLICIDAD**

En virtud del presente contrato, se establece que el "ARRENDADOR" podrá realizar actividades de publicidad en el inmueble arrendado, siempre y cuando se ajusten a las normas y regulaciones vigentes.

**ARTÍCULO 1: OBJETO DE LA CLÁUSULA**

La cláusula de publicidad tiene como objeto permitir al "ARRENDADOR" realizar actividades de publicidad en el inmueble arrendado, con el fin de promocionar sus productos o servicios.

**ARTÍCULO 2: CONDICIONES DE LA CLÁUSULA**

La cláusula de publicidad se regirá por las siguientes condiciones:

*   El "ARRENDADOR" podrá realizar actividades de publicidad en el inmueble arrendado, siempre y cuando no cause daño alguno al mismo ni a sus bienes.
*   Las actividades de publicidad deben ser realizadas de manera respetuosa con los demás ocupantes

# Nuevo LightRAG

In [ ]:
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.llms import Ollama
from pdf2image import convert_from_path
import pytesseract

# --- 1. CONFIGURACIÓN ---
# Usamos un modelo de embeddings que corre en tu CPU/GPU local, súper rápido.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
llm = Ollama(model="qwen2.5:latest") # O llama3

# --- 2. FUNCIÓN PARA CARGAR CONTRATOS ---
def cargar_contratos_a_vector_db():
    ruta = "./Contratos"
    documentos_finales = []
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

    for archivo in os.listdir(ruta):
        if archivo.endswith(".pdf"):
            print(f"📖 Leyendo {archivo}...")
            paginas = convert_from_path(os.path.join(ruta, archivo))
            texto = ""
            for p in paginas:
                texto += pytesseract.image_to_string(p, lang='spa')
            
            # Dividir en pedazos con metadatos del nombre del archivo
            chunks = splitter.split_text(texto)
            for chunk in chunks:
                documentos_finales.append({"text": chunk, "source": archivo})
    
    # Crear la base de datos vectorial
    texts = [d["text"] for d in documentos_finales]
    metadatos = [{"source": d["source"]} for d in documentos_finales]
    
    vector_db = Chroma.from_texts(
        texts=texts, 
        embedding=embeddings, 
        metadatas=metadatos,
        persist_directory="./db_vertiche" # Aquí se guarda todo
    )
    print("✅ Contratos indexados y listos para preguntas.")
    return vector_db

# --- 3. FUNCIÓN PARA PREGUNTAR ---
def preguntar_a_contratos(pregunta, vector_db):
    # Buscar los 5 pedazos más relevantes de los contratos
    docs = vector_db.similarity_search(pregunta, k=5)
    contexto = "\n\n".join([f"Archivo: {d.metadata['source']}\nContenido: {d.page_content}" for d in docs])
    
    prompt = f"""
    Eres un asistente legal experto de Vertiche. Usa el siguiente contexto de los contratos para responder la pregunta.
    Si no sabes la respuesta, dilo honestamente.

    En fechas o montos donde notes discrepancias, prioriza el dato escrito con LETRA sobre el número y escribe una recomendación de revisar manualmente el contrato.
    SIEMPRE transcribe nombres, fechas y montos tal cual aparecen, sin resumir ni interpretar.
    SIEMPRE menciona el nombre del artículo de donde sacaste la información.
    
    CONTEXTO:
    {contexto}
    
    PREGUNTA: {pregunta}
    """
    return llm.invoke(prompt)

In [23]:
import os
import json
import httpx
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pdf2image import convert_from_path
import pytesseract

# 1. Configuración de Modelos
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

async def llamar_ollama_json(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "qwen2.5:latest",
        "prompt": prompt,
        "format": "json", # Esto es clave
        "stream": False,
        "options": {
            "num_ctx": 8192, # Le damos más memoria
            "temperature": 0 # Lo hacemos más "serio" y preciso
        }
    }
    async with httpx.AsyncClient(timeout=180.0) as client:
        try:
            response = await client.post(url, json=payload)
            res_json = json.loads(response.json().get("response", "{}"))
            return res_json
        except Exception as e:
            print(f"⚠️ Error en Ollama: {e}")
            return {}

# 2. El Corazón del Sistema
async def procesar_auditoria_total():
    # --- NUEVA LÓGICA DE RUTAS ---
    # Esto obtiene la ruta exacta de la carpeta donde está tu proyecto
    directorio_base = os.path.abspath(os.getcwd())
    ruta_contratos = os.path.join(directorio_base, "Contratos")
    db_path = os.path.join(directorio_base, "db_vertiche_final")
    json_path = os.path.join(directorio_base, "auditoria_contratos.json")

    print(f"📂 Trabajando en: {directorio_base}")
    
    datos_estructurados = []
    documentos_para_db = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

    # Verificación de carpeta de entrada
    if not os.path.exists(ruta_contratos):
        print(f"❌ ERROR: No veo la carpeta {ruta_contratos}")
        return

    for archivo in os.listdir(ruta_contratos):
        if archivo.endswith(".pdf"):
            print(f"🧐 Analizando: {archivo}")
            ruta_pdf = os.path.join(ruta_contratos, archivo)
            
            # OCR y Extracción (Igual que antes)
            paginas = convert_from_path(ruta_pdf)
            texto_ocr = " ".join([pytesseract.image_to_string(p, lang='spa') for p in paginas])
            
            '''prompt_meta = f"De este contrato, extrae: Arrendador, Arrendatario, Renta Mensual, Vigencia (Inicio y Fin). Texto: {texto_ocr[:8000]}"
            metadatos = await llamar_ollama_json(prompt_meta)'''


            # --- DENTRO DE procesar_auditoria_total ---
            # A. EXTRAER METADATOS (Solo los primeros 4000 chars para no saturar)
            # Limpiamos el texto de saltos de línea locos que rompen el JSON
            texto_para_ia = " ".join(texto_ocr[:4000].split())
            
            prompt_meta = f"""
            Analiza este fragmento de contrato y extrae los datos en JSON. 
            Si no encuentras un dato, pon "No encontrado".
            Si hay discrepancias entre monto en letra y número, prioriza el de LETRA.
            
            Texto: {texto_para_ia}
            
            Responde con este formato exacto:
            {{
                "Arrendador": "Nombre",
                "Arrendatario": "Nombre",
                "Renta_Mensual": "Monto en numero",
                "Vigencia": "Fecha inicio a fin"
            }}
            """
            metadatos = await llamar_ollama_json(prompt_meta)
            metadatos['archivo_fuente'] = archivo
            datos_estructurados.append(metadatos)
            
            chunks = splitter.split_text(texto_ocr)
            for chunk in chunks:
                documentos_para_db.append({
                    "text": chunk,
                    "metadata": {"source": archivo, "arrendador": metadatos.get('Arrendador', 'Desconocido')}
                })

    # --- 3. GUARDADO REFORZADO ---
    print(f"Guardando JSON en: {json_path}")
    try:
        with open(json_path, "w", encoding='utf-8') as f:
            json.dump(datos_estructurados, f, indent=4, ensure_ascii=False)
        print("¡Archivo JSON creado físicamente!")
    except Exception as e:
        print(f"Error crítico al guardar JSON: {e}")
    
    # --- 4. CREAR DB ---
    vector_db = Chroma.from_texts(
        texts=[d["text"] for d in documentos_para_db],
        embedding=embeddings,
        metadatas=[d["metadata"] for d in documentos_para_db],
        persist_directory=db_path
    )
    
    return vector_db, datos_estructurados

'''# 5. Función para comparar o preguntar
def consulta_inteligente(pregunta, vector_db):
    # k=10 para traer suficiente contexto de varios archivos si es comparación
    docs = vector_db.similarity_search(pregunta, k=10)
    
    contexto = ""
    for d in docs:
        contexto += f"\n[ARCHIVO: {d.metadata['source']}]: {d.page_content}\n"
    
    prompt = f"""
    Eres un auditor experto de Vertiche. Responde basándote EXCLUSIVAMENTE en el contexto.
    Identifica claramente de qué contrato viene cada dato.
    
    CONTEXTO:
    {contexto}
    
    PREGUNTA: {pregunta}
    """
    # Aquí puedes llamar a Ollama para que genere la respuesta final
    print(f"Respuesta del Auditor:\n")'''

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


'# 5. Función para comparar o preguntar\ndef consulta_inteligente(pregunta, vector_db):\n    # k=10 para traer suficiente contexto de varios archivos si es comparación\n    docs = vector_db.similarity_search(pregunta, k=10)\n\n    contexto = ""\n    for d in docs:\n        contexto += f"\n[ARCHIVO: {d.metadata[\'source\']}]: {d.page_content}\n"\n\n    prompt = f"""\n    Eres un auditor experto de Vertiche. Responde basándote EXCLUSIVAMENTE en el contexto.\n    Identifica claramente de qué contrato viene cada dato.\n\n    CONTEXTO:\n    {contexto}\n\n    PREGUNTA: {pregunta}\n    """\n    # Aquí puedes llamar a Ollama para que genere la respuesta final\n    print(f"Respuesta del Auditor:\n")'

In [24]:
import os
print(f"📍 Tu carpeta de trabajo real es: {os.getcwd()}")
print(f"🔎 ¿Está el JSON aquí?: {'✅ SÍ EXISTE' if os.path.exists('auditoria_contratos.json') else '❌ NO ESTÁ'}")

📍 Tu carpeta de trabajo real es: c:\Users\Ame Contreras\Documents\AMEYALLI\1ITESM\8oSemestre\EmprendimientoTecnologico\Vertiche\1-Vertiche-BackEnd
🔎 ¿Está el JSON aquí?: ✅ SÍ EXISTE


In [25]:
# Esta es la línea que "enciende" todo el motor
v_db, info_json = await procesar_auditoria_total()

📂 Trabajando en: c:\Users\Ame Contreras\Documents\AMEYALLI\1ITESM\8oSemestre\EmprendimientoTecnologico\Vertiche\1-Vertiche-BackEnd
🧐 Analizando: C01_Contrato_Prueba-01.pdf
🧐 Analizando: C02_Contrato_Colima-02.pdf
Guardando JSON en: c:\Users\Ame Contreras\Documents\AMEYALLI\1ITESM\8oSemestre\EmprendimientoTecnologico\Vertiche\1-Vertiche-BackEnd\auditoria_contratos.json
¡Archivo JSON creado físicamente!


In [19]:
from langchain_community.llms import Ollama

In [26]:

# Cargamos el motor de embeddings (el mismo que usamos para crear la DB)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Cargamos la base de datos que ya existe en el disco
vector_db = Chroma(
    persist_directory="./db_vertiche_final", 
    embedding_function=embeddings
)

# Cargamos el JSON de metadatos para tenerlo a la mano
with open("auditoria_contratos.json", "r", encoding='utf-8') as f:
    data_estructurada = json.load(f)

# Inicializamos a Ollama para el chat
llm = Ollama(model="qwen2.5:latest") # o llama3

print(f"Sistema listo. Tenemos {len(data_estructurada)} contratos en el JSON y la DB cargada.")

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sistema listo. Tenemos 2 contratos en el JSON y la DB cargada.


In [27]:
async def auditor_inteligente(pregunta):
    # 1. Recuperar contexto de la DB
    docs = vector_db.similarity_search(pregunta, k=6)
    
    # 2. Formatear el contexto para que la IA sepa qué archivo es cada cual
    contexto_con_fuentes = ""
    for d in docs:
        contexto_con_fuentes += f"\n[ARCHIVO: {d.metadata['source']}]: {d.page_content}\n"
    
    # 3. Prompt técnico
    prompt = f"""
    Actúa como un Auditor Legal de Vertiche. 
    Responde la pregunta basándote únicamente en el contexto proporcionado.
    Es OBLIGATORIO mencionar el nombre del archivo al dar un dato.
    
    CONTEXTO:
    {contexto_con_fuentes}
    
    PREGUNTA: {pregunta}
    """
    
    # Usamos el modelo para generar la respuesta
    url = "http://localhost:11434/api/generate"
    payload = {"model": "qwen2.5:latest", "prompt": prompt, "stream": False}
    
    async with httpx.AsyncClient(timeout=120.0) as client:
        response = await client.post(url, json=payload)
        return response.json().get("response", "")

In [28]:
# PROBAR:
respuesta = await auditor_inteligente("¿Cuál es el monto de renta en Colima y quién es el arrendador?")
print(respuesta)

In [ ]:
# --- EJECUCIÓN ---
db = cargar_contratos_a_vector_db()

In [ ]:
respuesta = preguntar_a_contratos("¿Cuál es el plazo para avisar la terminación del contrato en Colima?", db)
print(respuesta)